# Injury news → `nfl.injury_news`

Reads FantasyPros injury snapshots landed by Azure `newsTrigger` (every 6h) under `injury/`,
explodes `items` with `lib.transforms.parse_injury_news`, and upserts by `id` (newest `snapshot_at` wins).
This table predates the ESPN tables and keeps its original audit column names (`source_file`, `snapshot_at`, `ingested_at`).

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

from lib import delta_io, schemas, transforms

ACCOUNT = "footballmanagerli"
TABLE = schemas.TABLES["injury_news"]
SRC = f"abfss://alsource@{ACCOUNT}.dfs.core.windows.net/{TABLE.source_prefix}/"
DEST = f"abfss://aldestination@{ACCOUNT}.dfs.core.windows.net/injury/injury_news"
CHECKPOINT = (
    f"abfss://aldestination@{ACCOUNT}.dfs.core.windows.net/_checkpoints/injury_news"
)

In [ ]:
delta_io.ensure_table(spark, TABLE, DEST)

raw = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")
    .schema(TABLE.raw_schema)
    .load(SRC)
)

parsed = transforms.parse_injury_news(transforms.with_file_metadata(raw))

(
    parsed.writeStream.foreachBatch(delta_io.upsert_batch_fn(spark, TABLE))
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .start()
    .awaitTermination()
)

In [ ]:
counts = spark.sql(
    f"SELECT COUNT(*) AS row_count, COUNT(DISTINCT id) AS key_count FROM {TABLE.full_name}"
).first()
print(counts)
assert counts.row_count == counts.key_count, "duplicate ids in table"

display(
    spark.sql(
        f"SELECT created_at, team_id, title, impact FROM {TABLE.full_name} ORDER BY created_at DESC LIMIT 20"
    )
)